In [1]:
import pandas as pd

X_train = pd.read_csv("../data/X_train.csv")

y_train = pd.read_csv(
    "../data/y_train.csv"
).squeeze()

In [2]:
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

In [3]:
from xgboost import XGBRegressor

from src.evaluate import rmse_cv

In [4]:
xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

xgb_rmse = rmse_cv(
    xgb_model,
    X_train,
    y_train
).mean()

print(
    f"XGBoost RMSE: {xgb_rmse:.5f}"
)

XGBoost RMSE: 0.11767


In [5]:
X_test = pd.read_csv(
    "../data/X_test.csv"
)

test = pd.read_csv(
    "../data/test.csv"
)

print(X_test.shape)
print(test.shape)

(1459, 313)
(1459, 80)


In [6]:
xgb_model.fit(
    X_train,
    y_train
)

print("XGB 訓練完成")

XGB 訓練完成


In [7]:
xgb_pred_log = xgb_model.predict(
    X_test
)

print(xgb_pred_log[:5])

[11.715387 12.048582 12.126418 12.157366 12.119203]


In [8]:
import numpy as np

xgb_pred = np.expm1(
    xgb_pred_log
)

print(xgb_pred[:5])

[122440.34 170855.97 184686.06 190491.06 183358.25]


In [9]:
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": xgb_pred
})

submission.head()

,Id,SalePrice
0,1461,122440.34375
1,1462,170855.96875
2,1463,184686.06250
3,1464,190491.06250
4,1465,183358.25000


In [10]:
submission.to_csv(
    "../submissions/xgb_submission.csv",
    index=False
)

print("XGB Submission 已建立")

XGB Submission 已建立


In [11]:
# XGBoost v2

xgb_model_v2 = XGBRegressor(
    n_estimators=3000,
    learning_rate=0.01,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_rmse_v2 = rmse_cv(
    xgb_model_v2,
    X_train,
    y_train
).mean()

print(
    f"XGB v2 RMSE: {xgb_rmse_v2:.5f}"
)

XGB v2 RMSE: 0.11427


In [12]:
xgb_model_v2.fit(
    X_train,
    y_train
)

print("XGB v2 訓練完成")

XGB v2 訓練完成


In [13]:
xgb_pred_log_v2 = xgb_model_v2.predict(
    X_test
)

print(xgb_pred_log_v2[:5])

[11.713678 12.001803 12.102861 12.166629 12.100695]


In [14]:
xgb_pred_v2 = np.expm1(
    xgb_pred_log_v2
)

print(xgb_pred_v2[:5])

[122231.266 163047.56  180386.28  192263.8   179995.86 ]


In [15]:
submission_v2 = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": xgb_pred_v2
})

submission_v2.head()

,Id,SalePrice
0,1461,122231.265625
1,1462,163047.562500
2,1463,180386.281250
3,1464,192263.796875
4,1465,179995.859375


In [16]:
submission_v2.to_csv(
    "../submissions/xgb_v2_submission.csv",
    index=False
)

print("XGB v2 Submission 已建立")

XGB v2 Submission 已建立


In [17]:
# XGBoost v3

xgb_model_v3 = XGBRegressor(
    n_estimators=5000,
    learning_rate=0.01,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_rmse_v3 = rmse_cv(
    xgb_model_v3,
    X_train,
    y_train
).mean()

print(
    f"XGB v3 RMSE: {xgb_rmse_v3:.5f}"
)

XGB v3 RMSE: 0.11446


In [18]:
xgb_model_v4 = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_rmse_v4 = rmse_cv(
    xgb_model_v4,
    X_train,
    y_train
).mean()

print(
    f"XGB v4 RMSE: {xgb_rmse_v4:.5f}"
)

XGB v4 RMSE: 0.11541


In [19]:
import numpy as np
from sklearn.model_selection import KFold, RandomizedSearchCV
from xgboost import XGBRegressor

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

xgb_base = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)

param_grid = {
    "n_estimators": np.arange(1500, 3500, 200),
    "learning_rate": [0.01, 0.015, 0.02, 0.025],
    "max_depth": [3, 4],
    "subsample": [0.75, 0.8, 0.85],
    "colsample_bytree": [0.75, 0.8, 0.85],
    "min_child_weight": [1, 2, 3],
}

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=10,   # 先用10
    scoring="neg_root_mean_squared_error",
    cv=kf,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

print("開始搜索...")

random_search.fit(
    X_train,
    y_train
)

print("\n=== 搜索完成 ===")
print(random_search.best_params_)
print(
    f"RMSE: {-random_search.best_score_:.5f}"
)

開始搜索...
Fitting 5 folds for each of 10 candidates, totalling 50 fits

=== 搜索完成 ===
{'subsample': 0.8, 'n_estimators': np.int64(1700), 'min_child_weight': 2, 'max_depth': 3, 'learning_rate': 0.025, 'colsample_bytree': 0.8}
RMSE: 0.11399


In [20]:
xgb_best = XGBRegressor(
    n_estimators=1700,
    learning_rate=0.025,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    random_state=42
)

xgb_best.fit(
    X_train,
    y_train
)

print("最佳 XGB 訓練完成")

最佳 XGB 訓練完成


In [21]:
xgb_best_pred_log = xgb_best.predict(
    X_test
)

xgb_best_pred = np.expm1(
    xgb_best_pred_log
)

In [22]:
submission_best = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": xgb_best_pred
})

submission_best.head()

,Id,SalePrice
0,1461,123159.640625
1,1462,159161.687500
2,1463,183050.218750
3,1464,195127.328125
4,1465,177529.187500


In [23]:
submission_best.to_csv(
    "../submissions/xgb_random_search_submission.csv",
    index=False
)

print("XGB Random Search Submission 已建立")

XGB Random Search Submission 已建立
